The data used to prepare this indicator can be found here https://tims.berkeley.edu/summary.php

Need to manually download data for the most recent 10 years under the Pedestrian and Bicyclist filters for both Fatalities and Fatalities/Serious Injuries.  Then, rename and save those workbooks according to the data already found in here https://sacog.sharepoint.com/sites/RegionalMonitoringandReporting/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FRegionalMonitoringandReporting%2FShared%20Documents%2FData%2FSafe%20Equitable%20Resilient%20Infrastructure%2FSafety%2FTIMS%2FBicyclists%20vs%20Pedestrians&viewid=8fba2a3f%2Dea8a%2D4663%2Da31f%2Dc1760cb6922d  

In [ ]:


# TIMS does not have an API available.  Data were manually downloaded from here https://tims.berkeley.edu/ and stored on SharePoint


import pandas as pd
import os
from pathlib import Path
import plotly.express as px
from IPython.display import display
import sys
sys.path.append(str(Path.cwd().parent.parent / 'config'))
import functions as func
import plot as pt


pd.set_option('display.max_columns', None)


# SharePoint OneDrive paths
PATH_SERVER = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data"
PATH_OUT  = r'C:\Users\jfontes\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Safe Equitable Resilient Infrastructure\Safety\Safety_4 Bicyclists vs Pedestrians'
FILE_ABOUT = r'C:\Users\jfontes\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Process Revamp\Task 6. Process Map\About Indicators.xlsx'
PATH_TIMS = Path(PATH_OUT).parent / 'TIMS' / 'Bicyclists vs Pedestrians'


EXPORT=False


def load_bike_ped(path_):

    def import_bike_ped(path_, geo, coll, cat):

        df = pd.read_csv(Path(path_)/coll/f'{geo}_{coll}_{cat}.csv')
        df[['Geography', 'Case', 'Type']] = geo, coll, cat
        df = (
                df[df['County']!='Statewide']
                    .drop('Average', axis=1)
                    .replace(',', '', regex=True)
            )

        return df


    dfs = {}
    for geo in geos:
        list_df_bike = []
        list_df_ped  = []
        for coll in collisions:
            for cat in categories:
                df = import_bike_ped(path_, geo, coll, cat)
                if cat == 'Bike':
                    list_df_bike.append(df)
                if cat == 'Ped':
                    list_df_ped.append(df)
        if geo == 'Counties':
            df_c_p = pd.concat(list_df_ped)
            df_c_b = pd.concat(list_df_bike)
        if geo == 'Jurisdictions':
            df_j_p = pd.concat(list_df_ped )
            df_j_b = pd.concat(list_df_bike)

    dfs['County_Ped' ] = df_c_p
    dfs['County_Bike'] = df_c_b
    dfs['City_Ped'   ] = df_j_p
    dfs['City_Bike'  ] = df_j_b

    return dfs


def reshape_bike_ped(df, geo_cat):

    if 'County' in geo_cat:
        geos = ['County']
    if 'City' in geo_cat:
        geos = ['County', 'City']
        df['City'] = df['City'].str.title()
    df['County'] = df['County'].str.title()
    geos = ['Geography', 'Type'] + geos

    df = pd.melt(df, id_vars=geos+['Case'], var_name='Year', value_name='Count')
    df['Count'] = df['Count'].astype('float32')
    df['Year'] = df['Year'].astype(int)
    df = df.pivot_table(index=geos+['Year'], columns='Case', values='Count').reset_index().drop(columns=['Geography'], errors='ignore')

    return df


def org_all_geos(dfs_bikeped):

    df_county = pd.concat([dfs_bikeped['County_Bike'], dfs_bikeped['County_Ped']])
    df_city   = pd.concat([dfs_bikeped['City_Bike'  ], dfs_bikeped['City_Ped'  ]])

    df_county = df_county.sort_values(['County'        , 'Year', 'Type'], ascending = [True, True, True])
    df_city   = df_city  .sort_values(['County', 'City', 'Year', 'Type'], ascending = [True, True, True, True])

    df_state = df_county[df_county['County'] == 'Statewide'].rename(columns = {'County':'Region'})
    df_county = df_county[df_county['County'] != 'Statewide']

    df_region = (
                df_county[df_county['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]
                        .drop(['County'], axis=1)
                        .groupby(['Year', 'Type'], as_index=False).sum()
        )
    df_region['Region'] = 'Sacramento Six-County Region'

    return df_city, df_county, df_region, df_state


def calc_5yr_avg(df, geo, geos):

    file_in = Path(PATH_OUT)/f'Safety_4 {geo} TIMS.xlsx'
    df_prev = pd.read_excel(file_in, sheet_name=geo)
    df_prev = df_prev[~df_prev['Year'].isin(df['Year'].unique())].drop(['Fatalities_5 Year Average', 'Serious Injuries_5 Year Average'], axis=1, errors='ignore') # To include year 2015 only

    df = pd.concat([df, df_prev], ignore_index=True)
    df["Type"] = df["Type"].replace({
            "Bike": "Bicyclists",
            "Ped": "Pedestrians"
        })

    df['Year'] = df['Year'].astype(int)
    df = df.sort_values(geos+['Year'], kind='mergesort').reset_index(drop=True)
    df['Fatalities_5 Year Average'      ] = df.groupby(geos+['Type'], sort=True)['Fatalities'      ].transform(lambda x: x.rolling(window=5, min_periods=5).mean())
    df['Serious Injuries_5 Year Average'] = df.groupby(geos+['Type'], sort=True)['Serious Injuries'].transform(lambda x: x.rolling(window=5, min_periods=5).mean())

    df = (
            df
                .sort_values(geos+['Year', 'Type'], ascending = [True]*len(geos)+[False, True])
                .set_index(geos+['Year', 'Type'])
                .reset_index()
        )

    return df




In [ ]:

geos = ['Counties', 'Jurisdictions']
collisions = ['Fatalities', 'Serious Injuries']
categories = ['Bike', 'Ped']

dfs_bikeped = load_bike_ped(PATH_TIMS)

for geo_cat, df in dfs_bikeped.items():
    dfs_bikeped[geo_cat] = reshape_bike_ped(df, geo_cat)

df_city, df_county, df_region, df_state = org_all_geos(dfs_bikeped)

df_city   = calc_5yr_avg(df_city  , 'Jurisdictions', ['County', 'City'])
df_county = calc_5yr_avg(df_county, 'Counties'     , ['County'])
df_region = calc_5yr_avg(df_region, 'Region'       , ['Region'])
df_state  = calc_5yr_avg(df_state , 'Statewide'    , ['Region'])

df_city = df_city.rename(columns = {'City':'Jurisdiction'})



In [ ]:


if EXPORT:

    # Update overall about documentations workbook
    sample_type = 'TIMS'
    indicator = 'Safety_4'
    year_start = df_county['Year'].min()
    year_end   = df_county['Year'].max()
    params = {
        'indicator': indicator
        , 'sample': sample_type
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
    }


    paths = [PATH_OUT, PATH_SERVER]

    dt_geo = {
                'Jurisdictions': df_city,
                'Counties': df_county,
                'Region': df_region,
                'Statewide': df_state,
            }
    
    for path in paths:
    
        for geography, df in dt_geo.items():
            params['geo']=geography
            df_about = func.write_about(params)
            file_out = Path(path) / f'{indicator} {geography} {sample_type}.xlsx'
            print(str(file_out))
            with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
                df_about.to_excel(writer, index=False, sheet_name='About', header=False)
                df      .to_excel(writer, index=False, sheet_name=geography            )




